# PROYECTO: Sistema de Evaluación de Riesgo Crediticio - FinTech Nova
## Data Analysis en Python

### Paso 1: Exploración de los Datos
**Objetivo:** Realizar un diagnóstico inicial del archivo crudo para comprender su estructura, el tipo de variables presentes y detectar anomalías o datos faltantes antes de la migración a la base de datos relacional.


In [6]:
!pip install pandas

In [7]:
import pandas as pd

# 1.1 Carga y dimensiones globales del dataset 
df = pd.read_csv('credit_risk_dataset.csv')

print('=== 1.1 DIMENSIONES DEL DATASET ===')
print(f"Total de Registros (filas): {df.shape[0]}")
print(f"Total de Columnas (columnas): {df.shape[1]}")

# 1.2 Inspecciones de tipos de datos por columna

print("\n=== 1.2 TIPOS DE DATOS POR COLUMNA ===")
# Se muestra una lista limpia de las columnas y cómo las interpreta Python
print(df.dtypes)

=== 1.1 DIMENSIONES DEL DATASET ===
Total de Registros (filas): 32581
Total de Columnas (columnas): 12

=== 1.2 TIPOS DE DATOS POR COLUMNA ===
person_age                      int64
person_income                   int64
person_home_ownership             str
person_emp_length             float64
loan_intent                       str
loan_grade                        str
loan_amnt                       int64
loan_int_rate                 float64
loan_status                     int64
loan_percent_income           float64
cb_person_default_on_file         str
cb_person_cred_hist_length      int64
dtype: object


### 1.3 Conteo y Porcentaje de Valores Nulos
**Objetivo:** Identificar qué columnas contienen celdas vacías y evaluar el impacto porcentual que tienen sobre el total del dataset para planificar la estrategia de limpieza.

In [8]:
# 1. Contar la cantidad absoluta de valores nulos por columna
valores_nulos = df.isnull().sum()

# 2. Calcular el porcentaje de nulos respecto al total de filas
porcentaje_nulos = (df.isnull().sum() / df.shape[0]) * 100

# 3. Combinar ambos resultados en una tabla limpia para analizar
tabla_nulos = pd.DataFrame({'Total Nulos': valores_nulos, 'Porcentaje(%)': porcentaje_nulos})

# 4. Mostrar solo las columnas que tengan al menos 1 valor nulo
print("=== COLUMNAS CON VALORES VACÍAS ===")
print(tabla_nulos[tabla_nulos['Total Nulos'] > 0])

=== COLUMNAS CON VALORES VACÍAS ===
                   Total Nulos  Porcentaje(%)
person_emp_length          895       2.747000
loan_int_rate             3116       9.563856


### 1.4 Resumen Estadístico y Valores Atípicos
**Objetivo:** Auditar los valores de las variables numéricas para identificar errores de carga humana o registros imposibles.

In [9]:
# Resumen estadístico de las variables numéricas
df.describe()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_cred_hist_length
count,32581.000000,3.258100e+04,31686.000000,32581.000000,29465.000000,32581.000000,32581.000000,32581.000000
mean,27.734600,6.607485e+04,4.789686,9589.371106,11.011695,0.218164,0.170203,5.804211
std,6.348078,6.198312e+04,4.142630,6322.086646,3.240459,0.413006,0.106782,4.055001
min,20.000000,4.000000e+03,0.000000,500.000000,5.420000,0.000000,0.000000,2.000000
25%,23.000000,3.850000e+04,2.000000,5000.000000,7.900000,0.000000,0.090000,3.000000
50%,26.000000,5.500000e+04,4.000000,8000.000000,10.990000,0.000000,0.150000,4.000000
75%,30.000000,7.920000e+04,7.000000,12200.000000,13.470000,0.000000,0.230000,8.000000
max,144.000000,6.000000e+06,123.000000,35000.000000,23.220000,1.000000,0.830000,30.000000


### Mis notas de Exploración de Tipos de Datos y Estructura:

1. Escala del Dataset: El mismo cuenta con 32581 registros y 12 columnas.
2. Interpretación de Python: Valida que los textos se leyeron como textos y los números como números.
3. Existencia de Nulos: Las únicas columnas con datos nulos son `person_emp_length` y `loan_int_rate`.

### Conclusiones del Diagnóstico Inicial (Paso 1):
1. **Calidad de Datos (Nulos):** Se detectaron vacíos en `person_emp_length` (2.74%) y `loan_int_rate` (9.56%). No se eliminarán debido a su alto volumen; se aplicarán técnicas de imputación.
2. **Errores Fácticos (Atípicos):** Se observan inconsistencias imposibles en los máximos de `person_age` (144 años) y `person_emp_length` (123 años). Estos registros serán eliminados por ser errores de origen.

### Paso 2: Limpieza y preparación de los datos
**Objetivo:** Corregir las anomalías detectadas en la fase de exploración. Se filtraran los registros imposibles y prepararemos el terreno para tratar los valores vacíos de forma que la base de datos quede 100% limpia para luego trabajar en PostgreSQL.


In [10]:
df[df['person_age'] >= 100]

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
81,144,250000,RENT,4.0,VENTURE,C,4800,13.57,0,0.02,N,3
183,144,200000,MORTGAGE,4.0,EDUCATION,B,6000,11.86,0,0.03,N,2
575,123,80004,RENT,2.0,EDUCATION,B,20400,10.25,0,0.25,N,3
747,123,78000,RENT,7.0,VENTURE,B,20000,NaN,0,0.26,N,4
32297,144,6000000,MORTGAGE,12.0,PERSONAL,C,5000,12.73,0,0.00,N,25


In [11]:
df[df['person_emp_length'] >= 100]

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
210,21,192000,MORTGAGE,123.0,VENTURE,A,20000,6.54,0,0.10,N,4


In [12]:
# 2.1 Filtramos y nos quedamos solo con los registros coherentes (o filas que tengan antigüedad vacía por ahora)
df_limpio = df.query("person_age <= 100 and (person_emp_length <= 60 or person_emp_length.isnull())")

print(f"Registros tras eliminar errores: {df_limpio.shape[0]}")

Registros tras eliminar errores: 32574


### 2.2 Tratamiento de Valores Nulos
**Estrategia:** 
1. Para `person_emp_length` (Antigüedad): Se imputarán los valores vacíos utilizando la **mediana** general, ya que la antigüedad suele estar concentrada en valores bajos y la mediana no se ve afectada por valores extremos.

2. Para `loan_int_rate` (Tasa de interés): Se aplicará una **imputación agrupada** utilizando el promedio de tasa correspondiente a cada categoría de riesgo (`loan_grade`).

In [15]:
# 1. Calcular la mediana de la antigüedad laboral
mediana_emp = df_limpio['person_emp_length'].median()
print(f'La mediana de antigüedad es: {mediana_emp} años')

# 2. Rellenar los valores nulos con la mediana
df_limpio['person_emp_length'] = df_limpio['person_emp_length'].fillna(mediana_emp)

# Comprobamos que ya no queden nulos en esa columna
print(f'Nulos restantes en person_emp_length: {df_limpio['person_emp_length'].isnull().sum()}')


La mediana de antigüedad es: 4.0 años
Nulos restantes en person_emp_length: 0


In [19]:
# 1. Calculamos el promedio agrupado por grado de préstamo y rellenamos los vacíos
df_limpio['loan_int_rate'] = df_limpio['loan_int_rate'].fillna(
    df_limpio.groupby('loan_grade')['loan_int_rate'].transform('mean')
)

# 2. Comprobación final de todo el dataset
print("=== COMPROBACIÓN FINAL DE NULOS ===")
print(df_limpio.isnull().sum())

=== COMPROBACIÓN FINAL DE NULOS ===
person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_grade                    0
loan_amnt                     0
loan_int_rate                 0
loan_status                   0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
dtype: int64
